# Measurement · validity  `[EVAL]`

**Is the ruler trustworthy?** The `arms/*`, `lookahead/*` and `method/*` families ask what the arms
did; this one asks whether the instrument that says so can be believed. Two questions, both read
from the `data/eval_scores/` lake that the paid `notebooks/scoring/Judge_Reliability.ipynb` writes
into — this notebook only *reads*, so it costs nothing and renders inside `render_results.py`.

- **§1 · Judge reliability** — the oracle's own repeatability (ICC), plus a decoupled second judge:
  do the endpoint contrasts keep their sign under a grader from a different model family that never
  played the patient?
- **§2 · Multi-judge** — where the variance in an arm mean actually comes from, whether gains
  transfer to a held-out grader, and at what effect size the two judges start agreeing.

> **Judge-invariant — which is why it is its own family.** Every artifact here contains *both*
> graders, so `reliability.py` loads them explicitly and ignores `EDA_JUDGE`. Exports go to
> `results/measurement/validity/{figures,tables}/` with **no `<judge>/` level**: a path naming one
> grader would assert that grader produced a cross-judge figure. `render_results.py` renders this
> notebook exactly once.
>
> **All 39 model states on one axis** (2026-08-18 reorg). The retired `L0`/`L5` views split the
> judged grid by K, which left the K=0 endpoint contrasts with `primary_n = 0` in the `L5` copy of
> `second_judge_contrasts` (the primary frame there held only K=5 arms). There is one view now; §1
> asserts every contrast row is paired on the full 96 personas.
>
> Split out of `5_Training_and_Reliability` on 2026-07-29 — that notebook is training-side and
> refuses a second judge, which forced these eval-side, cross-judge artifacts to be written under
> the primary oracle's folder.


In [ ]:
import sys, os
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 185, "display.max_columns", 50)

import os, eda_analysis
from eda_analysis import exports, plotting
from eda_analysis import reliability as rel
cfg = eda_analysis.EdaConfig(family="measurement/validity", judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()   # clears only THIS family's generated figures/tables (never SUMMARY.md)
# NOTE: no EDA_JUDGE handling, deliberately (a judge-invariant family: notebook_setup already
# ignores it). Every section reads EVERY judge from the score lake via reliability.py, so the
# active-judge knob would change nothing here; S.SCORES is used only to intersect the judged
# model states with the arms the config allows (default = every arm on disk).


## 1 · Judge reliability — oracle ICC + a decoupled second judge  `[EVAL]`

**Purpose.** Answer the two measurement-validity questions the rest of the EDA has to assume. Repeatability is measured on the anchor-model subset (base + endpoints + the GRPO peak × Q1/Q2/MICI × 96 convs); agreement runs on every cell both judges have scored — the full 39-state × 8-rubric grid. Scored by `Judge_Reliability.ipynb` (paid, run manually); this section only READS `data/eval_scores/`, so it stays free and inside `render_results.py`.

1. **Repeatability (LIMITATIONS §1).** The same oracle re-scoring the same conversations 3×, seeds differing and nothing else → **ICC(2,1)** + mean |Δ|. This is the instrument's own measurement error.
2. **Second judge (LIMITATIONS §2).** The simulated patient and the grading oracle are the same model (`gpt-4o-mini`), so the generator and evaluator are coupled. A different-family judge (Claude Haiku 4.5) that never played the patient breaks that coupling.

**Read.** Agreement is bounded by *both* raters' noise — compare `pearson_r` to the `ceiling` column, never to 1.0. The `bias` column is a LEVEL offset (a harsher judge marks everything down) and is irrelevant to the thesis, whose claims are all *contrasts*: the load-bearing panel is **contrast preservation** — if the PTO−GRPO endpoint gap keeps its sign under a decoupled judge, the headline is not an artifact of the shared patient/oracle model. `second_judge_contrasts` checks the two hand-picked K=0 endpoint pairs (`reliability.DEFAULT_CONTRAST_PAIRS`, paired on `file_index`, valid at matched iterations); every other pair — including the K=5 states — is in §2's persona-paired all-pairs table.


In [ ]:
# AUTO-DISCOVERED from what the second judge has actually scored — not a hardcoded subset.
# This section began life on a 4-model x 3-metric anchor subset; the full sweep covers all 39
# model states x 8 rubrics, and hardcoding would silently keep reporting the old corner of a grid
# that has since been completed.
if rel.available():
    _cov = rel.coverage_table(rel.load_judge_long(rel.second_judge_tags()[0]))
    JUDGE_METRICS = [m for m in eda_analysis.QUESTIONNAIRE_ORDER if m in set(_cov.metric)]
    JUDGE_MODELS = sorted(set(_cov.model))
else:
    JUDGE_METRICS, JUDGE_MODELS = [], []
_in_scope = [m for m in JUDGE_MODELS if m in set(S.SCORES.model)]
JUDGE_MODELS = _in_scope or JUDGE_MODELS          # respect the config's arm filter (default: every arm)
print(f"[judge] {len(JUDGE_MODELS)} model states x {len(JUDGE_METRICS)} metrics scored by the second judge")

if not rel.available():
    print("No re-scoring data on disk — run Judge_Reliability.ipynb first (writes data/eval_scores/judge=<tag>/rep=<r>/).")
elif not _in_scope:
    print("none of the judged model states is inside the config's arm filter — section skipped.")
else:
    TAG = rel.second_judge_tags()[0]
    JUDGE_NAME = rel.judge_display(TAG)

    # ── 1a · repeatability of the primary oracle ──────────────────────────────
    REP = rel.repeatability()
    if not REP.empty:
        display(rel.repeatability_by_metric(REP))
        exports.save_table(rel.repeatability_by_metric(REP), "oracle_repeatability_by_metric",
                           caption="Oracle repeatability per metric (primary oracle gpt-4o-mini, anchor-model subset): ICC(2,1) across 3 re-scorings of the same conversations (seeds differ only) + the mean per-conversation |delta| between reps. The citable 'oracle noise' figure.")
        exports.save_table(REP, "oracle_repeatability_icc",
                           caption="Oracle repeatability per (metric, model), primary oracle gpt-4o-mini: ICC(2,1) + mean |delta| across 3 re-scorings of the same 96 conversations.")
        fig = plotting.oracle_repeatability_bars(REP, metrics=JUDGE_METRICS)
        if fig:
            exports.save_fig(fig, "oracle_repeatability_icc",
                             caption="ICC(2,1) per model and metric from 3 re-scorings of the same conversations by the primary oracle (gpt-4o-mini); dotted = Koo & Li good (0.75) / excellent (0.90).")
            plt.show()

    # ── 1b · second judge vs the primary oracle ───────────────────────────────
    JL = rel.load_judge_long(TAG, reps=[0])
    PL = rel.load_primary_long(JUDGE_MODELS, JUDGE_METRICS)
    AGR = rel.agreement(JL, PL, REP)
    display(AGR)
    exports.save_table(AGR, "second_judge_agreement",
                       caption=f"Per-conversation agreement between {JUDGE_NAME} (held-out) and the primary oracle (gpt-4o-mini, the training reward), per (metric, model) over all {len(JUDGE_MODELS)} model states: Pearson r, Spearman rho, level bias (judge minus primary), and the attenuation ceiling implied by both judges' ICC.")
    fig = plotting.judge_agreement_scatter(JL, PL, agr_tab=AGR, metrics=JUDGE_METRICS,
                                           judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "judge_agreement_scatter",
                         caption=f"Per-conversation scores, {JUDGE_NAME} vs the primary oracle (gpt-4o-mini), one panel per metric, all model states pooled. Distance from the dashed identity line is level bias; scatter around a trend is rank disagreement.")
        plt.show()

    display(rel.arm_means_by_judge(JL, PL, JUDGE_NAME))

    # ── 1c · THE defense check: does the contrast survive the judge swap? ──────
    CON = rel.contrasts(JL, PL, JUDGE_METRICS)
    display(CON)
    # The one-view fix: with every arm in the primary frame, no contrast row may be unpaired.
    # (The retired L5 view rendered these K=0 pairs with primary_n = 0 and same_sign = False.)
    _bad = CON[(CON.primary_n == 0) | (CON.judge_n == 0)]
    assert _bad.empty, f"unpaired contrast rows (primary_n or judge_n == 0):\n{_bad}"
    print(f"[contrasts] {len(CON)} rows, primary_n {int(CON.primary_n.min())}-{int(CON.primary_n.max())}, "
          f"judge_n {int(CON.judge_n.min())}-{int(CON.judge_n.max())} — every row paired")
    exports.save_table(CON, "second_judge_contrasts",
                       caption=f"Contrast preservation: each hand-picked K=0 endpoint contrast (a minus b; MICI lower = better, so a negative delta favours a) as a paired delta over the 96 matched conversations (file_index pairing) under the primary oracle (gpt-4o-mini) and under {JUDGE_NAME}, with same_sign. The defense against the patient=oracle coupling in LIMITATIONS section 2. K=5 states and every other pair: multijudge_all_pairs_contrasts.")
    fig = plotting.judge_contrast_bars(CON, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "judge_contrast_preservation",
                         caption=f"K=0 endpoint contrasts under both judges (primary oracle gpt-4o-mini vs held-out {JUDGE_NAME}). Same-sign bars mean the result does not depend on the grader also having played the patient ({JUDGE_NAME} is a different model family). MICI: lower = better.")
        plt.show()

    print("\nVERDICT:", rel.summary_line(REP, AGR, CON))


## 2 · Multi-judge — variance sources, transfer, and resolution  `[EVAL]`

**Purpose.** §1 asked *does the contrast survive a second judge?* (yes, on two hand-picked pairs). This section asks the three follow-ups a defence actually turns on. Free — reads the same `data/eval_scores/` lake. Everything below runs over **every fully-scored (metric, model) cell** of all four arms — 39 model states (GRPO K=5 stops at iteration 5; the others run to 10).

**2a · Both judges, side by side.** A dumbbell per model state: bar *length* is the level offset, bar *order* is the claim. The two are never averaged — see the note above.

**2b · All pairwise contrasts.** Every pair of the 39 judged states × 8 rubrics — C(39,2) × 8 = 741 × 8 = 5,928 contrasts — under both judges. That is far too many to read as a table (the `.md` is a head excerpt; the leaf workbook holds every row), so the *rate* over it is what the thesis quotes, reported as a **sign-preservation ladder** by claimed effect size rather than a single pooled number. Pairing is on the recovered `persona_id`, not `file_index`: the trainer reshuffles the 96 personas every iteration, so a `file_index` join across unmatched iterations pairs unrelated conversations. Means are unaffected by that, but `dz` and the CI are not — and those are what a thesis table reports.

**2c · Variance decomposition.** Two-way random effects over arms × judges, on the arm means the thesis actually reports. Three components: **arm** (signal), **judge level** (large, and harmless — it cancels in every contrast), and **arm × judge** (the only one that threatens a claim: an ordering that depends on who is grading). `dependability_k1` is the generalizability coefficient for an arm mean read off a single judge — the number to quote when asked how far one judge's ranking can be trusted; `k2` is the same with both judges averaged, which is the honest answer to *"would a second judge help?"*.

**2d · Gain retention — the reward-hacking test.** What fraction of each arm's gain over Base survives the judge swap. Because the primary judge *was* the training reward and the second judge is held out, `Δ(judge) / Δ(primary)` is a **train/test generalization ratio**, not a reliability statistic: ~1.0 means the gain is a real behaviour change both judges see; ~0 means it lived only in the grader that was optimized. Uniform retention across arms is scale compression and uninteresting — the signal is retention that *differs by arm on one metric while staying flat on another*. The reference is ONE shared base draw (`PTOExp3_LA0_Base`, the long-standing convention) for all four arms; per-K and per-method references are the `lookahead/transfer` family's job.

With every iteration scored by both judges, retention is also a **trajectory**: reward hacking is a process, so the sharper question is not *"did this endpoint transfer?"* but ***"at which iteration did the gains stop transferring?"*** — a line declining with training estimates when the policy began fitting its grader, which no single-endpoint comparison can give you.

**2e · Concordance vs effect size.** *"When the primary judge reports a gap of at least x, how often does the second judge agree on the direction?"* — a curve, not a scalar, because a single r is dominated by the 1.2–1.7 point level offset that cancels in every contrast, while a rank statistic discards the magnitude that decides whether a gap matters. ⚠ **Each point is a pair of single conversations**; the thesis compares 96-conversation means, which resolve ~10× better. Do not read a bin height as confidence in an arm-level claim — that is what 2a and `dependability_k1` are for. Exact primary-judge ties are excluded (they state no ordering to reproduce; counting them as failures pushes the smallest bin below chance).


In [ ]:
if not rel.available() or not _in_scope:
    print("Multi-judge section skipped (no second-judge scores for the arms in scope).")
else:
    # The "gain over what?" baseline for 2d — derived from the judged frame, never hardcoded.
    # Prefer the PTO K=0 base to match the long-standing convention (one shared reference draw);
    # a hardcoded name absent from the frame once made gain_retention() skip every metric and
    # save an EMPTY multijudge_gain_retention.md (caught 2026-08-18).
    _bases = sorted(m for m in set(JL.model) & set(PL.model) if m.endswith("_Base"))
    _bases = [m for m in _bases if m.startswith("PTOExp3")] + _bases
    if not _bases:
        raise SystemExit("no *_Base model in the judged frame — cannot anchor gain retention")
    REFERENCE_MODEL = _bases[0]
    print(f"[retention] reference base = {REFERENCE_MODEL}")
    JUDGE_N_EXPECTED = 96                  # conversations per (metric, model) in a complete sweep

    # A second-judge sweep can land PARTIALLY (rate limits, expired batch, exhausted credit).
    # Partial cells are unbiased but less precise, and persona-paired stats collapse across two
    # partial arms — so restrict every table below to fully-scored cells and say what was dropped.
    COV = rel.coverage_table(JL, n_expected=JUDGE_N_EXPECTED)
    exports.save_table(COV, "multijudge_coverage",
                       caption=f"Conversations scored by {JUDGE_NAME} per (metric, model), out of {JUDGE_N_EXPECTED}, over every model state in the score lake. Section 2 analyses only cells marked complete; partial cells are reported here so a truncated sweep is never mistaken for full coverage.")
    if not COV.complete.all():
        print(f"[coverage] second-judge sweep is INCOMPLETE — "
              f"{int(COV.complete.sum())}/{len(COV)} cells fully scored "
              f"({COV.pct.min():.0f}-{COV.pct.max():.0f}% per cell). "
              f"Section 2 falls back to the complete cells only.")
    JL, PL = rel.filter_complete_cells(JL, PL, n_required=JUDGE_N_EXPECTED)
    JUDGE_METRICS = [m for m in JUDGE_METRICS if m in set(JL.metric)]
    if JL.empty or not JUDGE_METRICS:
        raise SystemExit("no fully-scored second-judge cells — nothing to analyse in section 2")

    # ── 2a · both judges side by side (never averaged) ────────────────────────
    fig = plotting.judge_dumbbell(JL, PL, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        # 39 model states per panel (the retired views held 22 / 17): the panel height the plotting
        # module picks is sized for the smaller grid and packs the y-tick labels into an unreadable
        # smear. Stretch the figure vertically and pull the header back down before saving.
        _w, _h = fig.get_size_inches()
        fig.set_size_inches(_w, _h * 2.2)
        if fig.legends:
            fig.legends[0].set_bbox_to_anchor((0.5, 1.02))
        if fig._suptitle is not None:
            fig._suptitle.set_y(1.045)
        fig.tight_layout()
        exports.save_fig(fig, "multijudge_arm_means_dumbbell",
                         caption=f"Arm means per model state (all four arms; GRPO K=5 ends at iteration 5) under the primary oracle (gpt-4o-mini) and {JUDGE_NAME}. Bar LENGTH is the level offset (large, and it cancels in every contrast); bar ORDER is what the thesis claims. Deliberately not averaged: the primary judge was the training reward and the second is held out, and the offset is model-dependent.")
        plt.show()

    # ── 2b · every pairwise contrast, persona-paired ──────────────────────────
    # Every pair of the judged model states x metric. C(39,2) x 8 = 5,928 rows: the .md is a head
    # excerpt (exports.MD_MAX_BYTES), the leaf workbook holds every row, and the sign-preservation
    # ladder below is the summary the thesis quotes.
    CONTRAST_MODELS = [m for m in JUDGE_MODELS if m in set(JL.model)]
    PAIRS = rel.all_pairs_contrasts(JL, PL, JUDGE_METRICS, models=CONTRAST_MODELS)
    print(f"[contrasts] {len(CONTRAST_MODELS)} model states -> {len(PAIRS)} contrasts "
          f"({len(set(JL.model))} states scored in total)")
    display(PAIRS[["metric", "contrast", "primary_delta", "judge_delta",
                   "judge_ci_lo", "judge_ci_hi", "judge_dz", "same_sign"]])
    exports.save_table(PAIRS, "multijudge_all_pairs_contrasts",
                       caption=f"Every model-state pair x metric ({len(CONTRAST_MODELS)} states, all four arms) under both judges, paired on the recovered persona (a minus b; MICI lower = better). judge_ci_* is a percentile bootstrap over personas (seed 42). same_sign is the defence: {int(PAIRS.same_sign.sum())}/{len(PAIRS)} contrasts keep their direction under {JUDGE_NAME}, which never played the patient. The .md is a head excerpt; the workbook holds every row.")

    # The rate over that table is what the thesis quotes, and it is only interpretable against an
    # effect size: a pooled "88% agree" reads as weak until you see the disagreements sit entirely
    # in gaps too small to claim. Save the ladder so the narrative docs cite a tracked artifact.
    SIGN = rel.sign_preservation(PAIRS)
    SIGN_BY_METRIC = rel.sign_preservation(PAIRS, by=["metric"])
    display(SIGN)
    exports.save_table(SIGN, "multijudge_sign_preservation",
                       caption=f"Share of the {len(PAIRS)} pairwise model-state x metric contrasts (all four arms) whose direction survives the swap to {JUDGE_NAME}, as a function of the gap the primary judge (gpt-4o-mini) reports. Read the row at the effect size you are claiming; the pooled row is dragged down by contrasts too small to claim in the first place.")
    exports.save_table(SIGN_BY_METRIC, "multijudge_sign_preservation_by_metric",
                       caption="The same ladder per rubric. A rubric that preserves sign less often is one whose arm ordering depends on who is grading - compare against dependability_k1 in multijudge_variance_components, which measures the same weakness from a completely different direction. WARNING: the thresholds are ABSOLUTE, so a row is comparable to other rows of the SAME rubric, never across rubrics - PCT and MICI live on a 0-1 scale and never reach 0.25, while Q1/Q2/WAI-SR/MITI are 1-5 or 1-7. The cross-rubric comparison is the all-contrasts row.")

    # ── 2c · where does arm-mean variance come from? ──────────────────────────
    VC_CONV = rel.variance_components_conversation(JL, PL, JUDGE_METRICS)
    VC_ARM = rel.variance_components_arm(JL, PL, JUDGE_METRICS, conv_components=VC_CONV)
    display(VC_ARM)
    exports.save_table(VC_ARM, "multijudge_variance_components",
                       caption=f"Two-way random-effects decomposition of the arm means the thesis reports (all {len(CONTRAST_MODELS)} judged model states x 2 judges): arm (signal) vs judge level (cancels in contrasts) vs arm x judge (ordering that depends on the grader). dependability_k1/k2 = generalizability of an arm mean read off one judge vs both averaged.")
    exports.save_table(VC_CONV, "multijudge_variance_components_per_conversation",
                       caption="The same decomposition at the per-conversation level, per (metric, model). var_resid here is per-conversation judge disagreement, which is what attenuates cross-judge correlations.")
    fig = plotting.variance_decomposition_bars(VC_ARM, metrics=JUDGE_METRICS)
    if fig:
        exports.save_fig(fig, "multijudge_variance_decomposition",
                         caption="Share of arm-mean variance by source (all judged model states, primary oracle gpt-4o-mini vs the held-out judge). A large judge-level slice is harmless (it cancels in contrasts); the arm x judge slice is the only component that threatens a claim.")
        plt.show()

    # ── 2d · does the improvement transfer to a held-out judge? ───────────────
    RET = rel.gain_retention(JL, PL, REFERENCE_MODEL, JUDGE_METRICS)
    display(RET)
    exports.save_table(RET, "multijudge_gain_retention",
                       caption=f"Fraction of each model state's gain over {REFERENCE_MODEL} (one shared reference draw for all four arms) that survives the swap to {JUDGE_NAME}. The primary judge (gpt-4o-mini) was the training reward and the second judge is held out, so this is a train/test generalization ratio: ~1.0 = a real behaviour change; ~0 = a gain that existed only in the optimized grader. Direction-agnostic (MICI lower = better flips both deltas). Persona-paired; CI is a persona bootstrap (seed 42); retention suppressed where |delta_primary| < 0.15.")
    fig = plotting.gain_retention_bars(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME)
    if fig:
        exports.save_fig(fig, "multijudge_gain_retention",
                         caption=f"Gain retention under a held-out judge ({JUDGE_NAME}), all four arms' model states, with persona-bootstrap CIs. Uniform bars across arms = scale compression; one arm collapsing while others hold = that arm's gain did not transfer.")
        plt.show()

    # Retention as a TRAJECTORY: with every iteration scored by both judges, "when did the gains
    # stop transferring?" becomes answerable, which no single-endpoint comparison can do.
    fig = plotting.retention_trajectory(RET, metrics=JUDGE_METRICS, judge_name=JUDGE_NAME,
                                        palette=S.PALETTE)
    if fig:
        exports.save_fig(fig, "multijudge_retention_trajectory",
                         caption=f"Gain retention vs iteration under {JUDGE_NAME}, one line per arm (GRPO K=5 ends at iteration 5), reference {REFERENCE_MODEL}. A line near 1.0 = gains a held-out judge also sees; a line declining with training = a policy progressively fitting the grader it was trained against, with the turn point estimating when that set in.")
        plt.show()

    # Q1-only single panel at column width. The retention claim in the reward-hacking draft rests
    # on the Q1 panel, and the full multi-metric grid is illegible at \columnwidth — this is the
    # figure the paper actually embeds (requested by the draft's Figure-3 legibility TODO).
    if "Q1" in set(RET.metric):
        fig = plotting.retention_trajectory(RET, metrics=["Q1"], ncols=1, judge_name=JUDGE_NAME,
                                            palette=S.PALETTE)
        if fig:
            exports.save_fig(fig, "multijudge_retention_trajectory_Q1",
                             caption=f"Gain retention vs iteration under {JUDGE_NAME}, Q1 only, all four arms — the single panel the retention claim rests on, sized for a one-column figure. Same data as the Q1 panel of multijudge_retention_trajectory.")
            plt.show()

    # ── 2e · how much resolution does a gap of a given size carry? ────────────
    CONC = pd.concat([rel.concordance_by_effect_size(JL, PL, m, scope=s)
                      for m in JUDGE_METRICS for s in ("cross_model", "within_model")],
                     ignore_index=True)
    if not CONC.empty:
        display(CONC.pivot_table(index="bin", columns=["metric", "scope"], values="concordance"))
        exports.save_table(CONC, "multijudge_concordance_by_effect_size",
                           caption=f"P({JUDGE_NAME} agrees on the direction) as a function of the gap the primary judge (gpt-4o-mini) reports, per conversation PAIR (seeded sample of at most 400,000 pairs over all judged model states). Exact primary-judge ties excluded. Not a confidence in any arm-level claim - arm means over 96 conversations resolve far better; see multijudge_all_pairs_contrasts.")
        fig = plotting.concordance_curve(CONC, judge_name=JUDGE_NAME)
        if fig:
            exports.save_fig(fig, "multijudge_concordance_curve",
                             caption=f"Cross-judge ordering agreement ({JUDGE_NAME} vs the primary oracle) vs effect size, per conversation pair. Shows how much per-conversation resolving power a given gap carries - i.e. why 96 conversations per arm are needed.")
            plt.show()

    print("\nVERDICT:", rel.multi_judge_summary_line(VC_ARM, RET, PAIRS))

    # ── ledger · the handful of citable scalars, keyed for the papers ─────────
    # reliability.py ships no *_numbers() builder, so the ledger is assembled here from the tables
    # just saved (every value names its table). Point estimates only — no bootstrap CI keys.
    NUM = {}
    if not REP.empty:
        for r in rel.repeatability_by_metric(REP).itertuples():
            NUM[f"oracle_icc.{r.metric}.icc_2_1"] = {"value": float(r.icc_2_1), "source": "tables/oracle_repeatability_by_metric.md",
                                                     "note": "primary oracle ICC(2,1), mean over anchor models"}
            NUM[f"oracle_icc.{r.metric}.mean_abs_diff"] = {"value": float(r.mean_abs_diff), "source": "tables/oracle_repeatability_by_metric.md",
                                                           "note": "mean per-conversation |delta| between re-scorings"}
    for r in CON.itertuples():
        key = f"contrast.{r.contrast.replace(' − ', '_minus_').replace(' ', '')}.{r.metric}"
        NUM[f"{key}.primary_delta"] = {"value": float(r.primary_delta), "source": "tables/second_judge_contrasts.md", "note": "gpt-4o-mini, file_index-paired, n=%d" % r.primary_n}
        NUM[f"{key}.judge_delta"] = {"value": float(r.judge_delta), "source": "tables/second_judge_contrasts.md", "note": f"{JUDGE_NAME}, file_index-paired, n=%d" % r.judge_n}
        NUM[f"{key}.same_sign"] = {"value": bool(r.same_sign), "source": "tables/second_judge_contrasts.md", "note": "sign preserved under the held-out judge"}
    NUM["contrast.n_same_sign"] = {"value": int(CON.same_sign.sum()), "source": "tables/second_judge_contrasts.md", "note": f"of {len(CON)} hand-picked K=0 endpoint contrasts"}
    NUM["contrast.n_total"] = {"value": int(len(CON)), "source": "tables/second_judge_contrasts.md", "note": "rows in second_judge_contrasts"}
    for r in SIGN.itertuples():
        sub = r.subset.replace("|Δ primary| ≥ ", "abs_primary_ge_").replace(" ", "_").replace(".", "p")
        NUM[f"sign_preservation.{sub}.pct_same_sign"] = {"value": float(r.pct_same_sign), "source": "tables/multijudge_sign_preservation.md",
                                                          "note": f"{int(r.n_same_sign)}/{int(r.n_contrasts)} persona-paired contrasts, all four arms"}
    for r in VC_ARM.itertuples():
        NUM[f"variance.{r.metric}.pct_arm_x_judge"] = {"value": float(r.pct_arm_x_judge), "source": "tables/multijudge_variance_components.md", "note": "share of arm-mean variance in the arm x judge interaction"}
        NUM[f"variance.{r.metric}.dependability_k1"] = {"value": float(r.dependability_k1), "source": "tables/multijudge_variance_components.md", "note": "generalizability of an arm mean read off ONE judge"}
    NUM["multijudge.n_states"] = {"value": int(len(CONTRAST_MODELS)), "source": "tables/multijudge_coverage.md", "note": "model states with complete second-judge coverage"}
    NUM["multijudge.n_pairs_contrasts"] = {"value": int(len(PAIRS)), "source": "tables/multijudge_all_pairs_contrasts.md", "note": "C(n_states,2) x n_metrics"}
    NUM["multijudge.reference_model"] = {"value": REFERENCE_MODEL, "source": "tables/multijudge_gain_retention.md", "note": "gain-retention reference draw"}
    exports.save_numbers("validity_numbers", NUM,
                         caption="Citable scalars of this family (oracle ICC per metric, the hand-picked K=0 endpoint contrasts under both judges, the sign-preservation ladder, per-metric arm x judge share + dependability_k1); every value names the table it was read from.")
    print(f"[ledger] validity_numbers: {len(NUM)} keys")


## 3 · How to read this notebook
- **ICC (§1)** is how much of a per-conversation score is signal rather than re-scoring noise. Read it against Koo & Li (2016): ≥0.75 good, ≥0.90 excellent. It bounds everything downstream — an arm difference smaller than the grader's own noise is not a difference.
- **Cross-judge `r` must be compared to the `ceiling`, never to 1.0.** The ceiling is `sqrt(ICC_primary × ICC_judge)`: two imperfect raters cannot correlate perfectly even when measuring the same thing. Both terms have been measured since 2026-07-28; `ceiling_basis` records whether a cell used measured values or fell back to the old `ICC_judge == ICC_primary` assumption.
- **`same_sign` is the load-bearing number**, not the correlation. A large level `bias` between judges is expected and harmless — the thesis reports contrasts, which cancel it. What would hurt is an arm *ordering* that depends on who grades.
- **Never average the two judges' raw scores.** The primary oracle *was the training reward*; the second judge never touched training. That is optimization-target vs held-out-test, not two interchangeable raters — `reliability.py` enforces this and only ever combines contrasts or standardized quantities.
- **`arm × judge` (§2) is the only variance component that can invalidate a claim.** A large `judge` term is a level shift; a large interaction means the ranking itself moves with the grader. Read `dependability_k1` as "how far can I trust an arm ranking taken off ONE judge".
- **Gain retention (§2) is the reward-hacking test**: `Δ(held-out) / Δ(trained-against)`. ~1.0 = a real behaviour change both graders see; ~0 = a gain that existed only in the optimized grader.
- **Concordance (§2) is per conversation PAIR, not per arm** — arm means over 96 conversations resolve ~10× better, so do not read a bin height as confidence in an arm-level claim.
- _(The measured values are narrated in `results/measurement/SUMMARY.md` and the caveats they imply in `results/LIMITATIONS.md` §1–§3. This notebook is where they are computed.)_


In [ ]:
exports.prune_orphan_captions(); print("index ->", exports.build_index())